# QICK CW Calibration — Gain Curve Measurement
### RFSoC 4x2 · QICK overlay · University of Manchester / JBO
**Goal:** Sweep the QICK DAC across the RHINO science band (60–85 MHz),  
measure received power at each frequency step, and extract the frequency-  
dependent gain curve g(ν) of the signal chain.

**Hardware:** DAC_B → SMA loopback cable → ADC_D  
**Why QICK:** True direct sampling — no DDC lock, no NCO spur, no F_LO offset.  
Tone at f_DAC appears at exactly f_DAC in the ADC spectrum.

**Run all cells top-to-bottom. Each cell prints  PASS or  FAIL.**

---

## Cell 1 — Imports and environment check

In [ ]:
import sys, os, time, datetime
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

REQUIRED = ['numpy', 'matplotlib', 'qick']
missing  = []
for pkg in REQUIRED:
    try:
        __import__(pkg)
        print(f'   {pkg} importable')
    except ImportError:
        print(f'   {pkg} NOT FOUND')
        missing.append(pkg)

print(f'\nPython     : {sys.version.split()[0]}')
print(f'NumPy      : {np.__version__}')
print(f'Matplotlib : {matplotlib.__version__}')

if missing:
    print(f'\n FAIL — missing packages: {missing}')
else:
    print('\n PASS — all packages available')

## Cell 2 — Load QICK overlay
**Takes ~60 seconds. The FPGA is being programmed.**

In [ ]:
from qick import QickSoc

BIT_FILE = '/home/xilinx/qick_repo/qick_lib/qick/qick_4x2.bit'
HWH_FILE = '/home/xilinx/qick_repo/qick_lib/qick/qick_4x2.hwh'

for fpath in [BIT_FILE, HWH_FILE]:
    if os.path.exists(fpath):
        print(f'   Found: {os.path.basename(fpath)}')
    else:
        print(f'   MISSING: {fpath}')

print('\n[LOAD] Programming FPGA — please wait ~60 seconds...\n')
t0  = time.time()
soc = QickSoc(bitfile=BIT_FILE)
soc.config = soc.get_cfg()  # API fix: patch config dict for QICK 0.2.388
print(f'\n[LOAD] Done in {time.time()-t0:.0f}s')
print(soc)
print('\n PASS — QICK overlay loaded')

## Cell 3 — Hardware configuration
Confirms ADC/DAC sample rates match the validated configuration.

In [ ]:
EXPECTED_ADC_FS  = 4423.680   # MHz — confirmed on board
EXPECTED_DAC_FS  = 9830.400   # MHz
FS_TOL           = 1.0         # MHz tolerance

# API fix: QICK 0.2.388 uses get_cfg() not soc.config
soc.config = soc.get_cfg()
adc_fs = soc.config['readouts'][0]['fs']
dac_fs = soc.config['gens'][0]['fs']

passed = True
for label, measured, expected in [
    ('ADC sample rate', adc_fs, EXPECTED_ADC_FS),
    ('DAC sample rate', dac_fs, EXPECTED_DAC_FS),
]:
    ok = abs(measured - expected) < FS_TOL
    symbol = '✅' if ok else '❌'
    print(f'  {symbol} {label}: {measured:.3f} MHz  (expected {expected:.3f} MHz)')
    if not ok:
        passed = False

FS_MHZ      = adc_fs
NYQUIST_MHZ = FS_MHZ / 2.0
print(f'\n  ADC Nyquist : {NYQUIST_MHZ:.2f} MHz')
print(f'  Direct sampling — no DDC, no NCO spur, no F_LO offset')
# Note: NYQUIST_DEC_MHZ is defined in Cell 4 (config). Range check is done there.
print(f'  Full ADC Nyquist: {NYQUIST_MHZ:.1f} MHz')
print(f'\n{" PASS" if passed else " FAIL"} — hardware configuration')

## Cell 4 — Experiment configuration
Edit the values in this cell to match your setup.  
All subsequent cells use these parameters — do not edit them elsewhere.

| Parameter | Loopback (bench) | Signal generator (JBO) |
|---|---|---|
| `SWEEP_START_MHZ` | 60 | 60 |
| `SWEEP_STOP_MHZ` | 85 | 85 |
| `N_STEPS` | 27 | 27 |
| `N_FRAMES_PER_STEP` | 200 | 200 |
| `N_BASELINE_FRAMES` | 500 | 500 |
| `FFT_SIZE` | 8192 | 65536 |

In [ ]:
import math

# ── Hardware channels ─────────────────────────────────────────────────────────
ADC_CH    = 0   # readout ch 0 = ADC_D SMA
DAC_GEN   = 0   # signal generator ch 0 = DAC_B SMA

# ── Spectral parameters ───────────────────────────────────────────────────────
READOUT_LENGTH  = 1000
WINDOW_FUNCTION = 'hann'

FS_DECIMATED_MHZ = soc.config['readouts'][ADC_CH]['f_fabric']   # 552.960 MHz confirmed
NYQUIST_DEC_MHZ  = FS_DECIMATED_MHZ / 2.0
DF_MHZ           = FS_DECIMATED_MHZ / READOUT_LENGTH            # 0.55296 MHz per channel
DF_KHZ           = DF_MHZ * 1e3

# ── Jordan's channel-centre formula ──────────────────────────────────────────
def get_channel_centres(fs_mhz, fft_size, n_divisions,
                        start_mhz=None, stop_mhz=None):
    """
    Return (channel_indices, centre_frequencies_mhz) for a CW calibration sweep.

    The band (0 to fs_mhz/2) is divided into fft_size//2 channels each of width
    df = fs_mhz / fft_size MHz. Channel k spans [k*df, (k+1)*df]; its centre is
    (k + 0.5)*df.

    n_divisions : int, must be a power of 2.
        Determines how many channels are sampled:
            n_divisions=1  → every channel  (channel_step = 1)
            n_divisions=2  → every 2nd      (channel_step = 2)
            n_divisions=8  → every 8th      (channel_step = 8)  etc.
        channel_step = n_divisions (step between selected channel indices).

    start_mhz, stop_mhz : optional float
        Restrict output to channels whose centres fall within [start_mhz, stop_mhz].
        If None, covers the full 0 to Nyquist band.

    Returns
    -------
    channel_indices   : np.ndarray of int   — bin index k for each selected channel
    centre_freqs_mhz  : np.ndarray of float — centre frequency (k+0.5)*df in MHz
    """
    if n_divisions < 1 or (n_divisions & (n_divisions - 1)) != 0:
        raise ValueError(f'n_divisions must be a power of 2, got {n_divisions}')

    df         = fs_mhz / fft_size          # MHz per channel
    n_channels = fft_size // 2              # total channels 0 to Nyquist
    step       = n_divisions                # channel_step = n_divisions

    all_k      = np.arange(0, n_channels, step)
    all_f      = (all_k + 0.5) * df        # centre frequencies in MHz

    # Apply optional band restriction
    mask = np.ones(len(all_k), dtype=bool)
    if start_mhz is not None:
        mask &= (all_f >= start_mhz)
    if stop_mhz is not None:
        mask &= (all_f <= stop_mhz)

    return all_k[mask].astype(int), all_f[mask]


# ── Sweep mode ────────────────────────────────────────────────────────────────
# MODE A — broad sweep (every Nth channel across band)
#   n_divisions = channel_step (power of 2: 1, 2, 4, 8, 16, 32 ...)
#   start/stop in MHz restrict to the band of interest
#
# MODE B — narrow sweep (fixed MHz range, fixed number of steps)
#   Measures sub-channel response shape: place 32 tones across e.g. 50.0-50.1 MHz
#   to see how the channel response function looks (centre bright, neighbours zero)

SWEEP_MODE = 'broad'   # 'broad' or 'narrow'

# ── Broad sweep parameters ────────────────────────────────────────────────────
N_DIVISIONS    = 8      # power of 2 — every 8th channel
SWEEP_START_MHZ = 50.0
SWEEP_STOP_MHZ  = 200.0

# ── Narrow sweep parameters (channel response validation) ────────────────────
CW_START_FREQ  = 50.0    # MHz
CW_END_FREQ    = 50.6    # MHz  (just over 1 channel width = 0.553 MHz)
CW_STEPS       = 32      # must be power of 2

# ── Acquisition ───────────────────────────────────────────────────────────────
N_FRAMES_PER_STEP = 200
N_BASELINE_FRAMES = 500
DAC_AMPLITUDE     = 0.9

# ── Build tone array from selected mode ───────────────────────────────────────
if SWEEP_MODE == 'broad':
    ch_indices, DAC_FREQS_MHZ = get_channel_centres(
        FS_DECIMATED_MHZ, READOUT_LENGTH, N_DIVISIONS,
        start_mhz=SWEEP_START_MHZ, stop_mhz=SWEEP_STOP_MHZ)
    print(f'[CONFIG] Broad sweep: every {N_DIVISIONS} channels')
elif SWEEP_MODE == 'narrow':
    # Narrow sweep uses evenly spaced frequencies (sub-channel resolution)
    DAC_FREQS_MHZ = np.linspace(CW_START_FREQ, CW_END_FREQ, CW_STEPS)
    ch_indices    = np.array([int(f / DF_MHZ) for f in DAC_FREQS_MHZ])
    print(f'[CONFIG] Narrow sweep: {CW_START_FREQ}-{CW_END_FREQ} MHz, {CW_STEPS} steps')
    print(f'  Step size   : {(CW_END_FREQ-CW_START_FREQ)/(CW_STEPS-1)*1e3:.2f} kHz')
    print(f'  Channel width: {DF_KHZ:.2f} kHz  →  this sweep spans '
          f'{(CW_END_FREQ-CW_START_FREQ)/DF_MHZ:.2f} channels')
else:
    raise ValueError(f"SWEEP_MODE must be 'broad' or 'narrow', got '{SWEEP_MODE}'")

N_STEPS = len(DAC_FREQS_MHZ)

# ── Frequency axis and window ─────────────────────────────────────────────────
freq_axis_mhz = np.fft.rfftfreq(READOUT_LENGTH,
                                  d=1.0/(FS_DECIMATED_MHZ*1e6)) / 1e6
if WINDOW_FUNCTION == 'hann':
    window = np.hanning(READOUT_LENGTH).astype(np.float32)
elif WINDOW_FUNCTION == 'rect':
    window = np.ones(READOUT_LENGTH, dtype=np.float32)
else:
    window = np.blackman(READOUT_LENGTH).astype(np.float32)
window_power = float(np.sum(window**2))

# ── Output ────────────────────────────────────────────────────────────────────
SAVE_DIR = '/home/xilinx/jupyter_notebooks/cw_calibration/'
import os; os.makedirs(SAVE_DIR, exist_ok=True)

# ── DAC power estimate ────────────────────────────────────────────────────────
V_rms_est = (DAC_AMPLITUDE * 1.0) / (2 * math.sqrt(2))
P_dBm_est = 10 * math.log10(V_rms_est**2 / 50.0 / 1e-3)

print(f'  fs_decimated   : {FS_DECIMATED_MHZ:.3f} MHz')
print(f'  Channel width  : {DF_KHZ:.3f} kHz')
print(f'  N steps        : {N_STEPS}')
print(f'  First tone     : {DAC_FREQS_MHZ[0]:.4f} MHz  (ch {ch_indices[0]})')
print(f'  Last tone      : {DAC_FREQS_MHZ[-1]:.4f} MHz  (ch {ch_indices[-1]})')
print(f'  DAC amplitude  : {DAC_AMPLITUDE} → est. {P_dBm_est:.1f} dBm into 50 Ω')
print(f'  Frames/step    : {N_FRAMES_PER_STEP}')
# Convenience variables used in plot cells — work in both broad and narrow mode
SWEEP_BAND_START = float(DAC_FREQS_MHZ[0])
SWEEP_BAND_END   = float(DAC_FREQS_MHZ[-1])
SWEEP_SPAN       = max(SWEEP_BAND_END - SWEEP_BAND_START, DF_MHZ * 2)

# ── Board spur flagging (Jordan's request) ───────────────────────────────────
# The 100.1 MHz board artefact corrupts baseline subtraction near 93-111 MHz.
# These channels are flagged as unreliable and skipped in the gain curve.
SPUR_LO_MHZ = 93.0    # lower edge of corrupted region
SPUR_HI_MHZ = 111.0   # upper edge of corrupted region
spur_mask   = (DAC_FREQS_MHZ >= SPUR_LO_MHZ) & (DAC_FREQS_MHZ <= SPUR_HI_MHZ)
valid_mask  = ~spur_mask
n_flagged   = int(np.sum(spur_mask))
print(f'  Board spur region: {SPUR_LO_MHZ}-{SPUR_HI_MHZ} MHz  ')
print(f'  Flagged steps     : {n_flagged}/{N_STEPS}  '
      f'({[f"{f:.1f}" for f in DAC_FREQS_MHZ[spur_mask]]})')
print(f'  Valid steps       : {int(np.sum(valid_mask))}/{N_STEPS}')

print('\n✅ Configuration ready')


## Cell 5 — ADC capture function
Defines how to get a single raw ADC spectrum. Uses decimated buffer for speed; adapt to DDR4 for high resolution.

In [ ]:
from qick.averager_program import AveragerProgram

class NoToneProgram(AveragerProgram):
    """Captures ADC samples with DAC silent. Used for baseline only."""
    def initialize(self):
        self.declare_readout(
            ch=self.cfg['ro_ch'], length=self.cfg['readout_length'],
            freq=0, gen_ch=None)
        self.synci(200)

    def body(self):
        self.trigger(adcs=[self.cfg['ro_ch']], pins=[0],
                     adc_trig_offset=self.cfg['adc_trig_offset'])
        self.wait_all()
        self.sync_all(self.us2cycles(self.cfg['relax_delay']))

_no_tone_prog = NoToneProgram(soc, {
    'ro_ch'          : ADC_CH,
    'readout_length' : READOUT_LENGTH,
    'adc_trig_offset': 200,
    'soft_avgs'      : 1,
    'reps'           : 1,
    'relax_delay'    : 1.0,
})

def compute_spectrum_db(samples):
    """
    Apply window and compute power spectrum in dB.
    samples must be exactly READOUT_LENGTH long.
    Returns array of length READOUT_LENGTH//2 + 1.
    """
    if len(samples) != READOUT_LENGTH:
        samples = samples[:READOUT_LENGTH] if len(samples) > READOUT_LENGTH                   else np.pad(samples, (0, READOUT_LENGTH - len(samples)))
    windowed = samples * window
    power    = np.abs(np.fft.rfft(windowed))**2 / window_power
    return (10.0 * np.log10(np.maximum(power, 1e-30))).astype(np.float32)

def capture_baseline_sample():
    """One spectrum with DAC off. Returns dB array of length READOUT_LENGTH//2+1."""
    iq     = _no_tone_prog.acquire_decimated(soc, load_pulses=False, progress=False)
    i_data = np.array(iq[0][0], dtype=np.float32)
    return compute_spectrum_db(i_data)

print(f'[SETUP] Baseline capture ready')
print(f'  Readout length   : {READOUT_LENGTH} samples at {FS_DECIMATED_MHZ:.3f} MHz')
print(f'  Freq resolution  : {DF_KHZ:.2f} kHz/bin')
print(f'  Freq axis bins   : {len(freq_axis_mhz)}  (0 to {freq_axis_mhz[-1]:.1f} MHz)')
print(f'  Trigger mode     : single trigger per body() rep, soft_avgs=1')
print('\n✅ PASS — baseline capture function ready')


## Cell 5b — RFI baseline spectrum (no CW)
Jordan's request: capture spectra with no CW tone, same sampling and decimation,
to inspect the band for RFI before the calibration sweep.
**Run AFTER Cell 5 (capture_baseline_sample is now defined). DAC silent, loopback cable connected.**
Save and download the plot to share with Jordan and Phil.

In [ ]:
# ── RFI baseline: 500-frame average spectrum with no CW tone ──────────────────
# This is what Jordan asked for: same sampling/decimation, no CW, inspect the band.
# Requires Cell 5 (NoToneProgram + capture_baseline_sample) to have been run first.

print(f'[RFI BASELINE] Capturing {N_BASELINE_FRAMES} frames with DAC off...')
disable_tone() if '_active_prog' in dir() else None
import time
time.sleep(0.5)

rfi_acc = np.zeros(len(freq_axis_mhz), dtype=np.float64)
for k in range(N_BASELINE_FRAMES):
    rfi_acc += capture_baseline_sample().astype(np.float64)
    if (k+1) % 100 == 0:
        print(f'  Frame {k+1}/{N_BASELINE_FRAMES}...')

rfi_spec = (rfi_acc / N_BASELINE_FRAMES).astype(np.float32)
rfi_mean = float(np.mean(rfi_spec))
rfi_std  = float(np.std(rfi_spec))

print(f'\n--- RFI baseline summary ---')
print(f'  Mean power   : {rfi_mean:.2f} dB')
print(f'  Std dev      : {rfi_std:.2f} dB')
print(f'  Dynamic range: {float(np.max(rfi_spec)) - rfi_mean:.2f} dB')
print(f'  Peak at      : {freq_axis_mhz[int(np.argmax(rfi_spec))]:.2f} MHz  ({float(np.max(rfi_spec)):.1f} dB)')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(freq_axis_mhz, rfi_spec - rfi_mean, color='#00e5ff', lw=0.8, alpha=0.9)
ax.axhline(0, color='grey', lw=0.6, linestyle='--', label='Mean floor')
ax.axvspan(50, 200, alpha=0.07, color='cyan', label='50–200 MHz sweep band')
ax.set_xlim(0, NYQUIST_DEC_MHZ)
ax.set_xlabel(f'Frequency (MHz)  [{DF_KHZ:.2f} kHz/bin]', fontsize=11)
ax.set_ylabel('Power relative to mean (dB)', fontsize=11)
ax.set_title(
    f'RFI baseline spectrum — no CW tone  |  {N_BASELINE_FRAMES} frames averaged\n'
    f'QICK direct sampling  |  fs_dec = {FS_DECIMATED_MHZ:.3f} MHz  |  Hann window',
    fontsize=10
)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)
ax.set_ylim(-15, 20)

rfi_ts  = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
rfi_out = f'{SAVE_DIR}rfi_baseline_{rfi_ts}.png'
plt.savefig(rfi_out, dpi=150, bbox_inches='tight')
plt.show()

np.save(f'{SAVE_DIR}rfi_baseline_{rfi_ts}.npy', rfi_spec)
np.save(f'{SAVE_DIR}rfi_freq_axis_{rfi_ts}.npy', freq_axis_mhz)
print(f'\n  Saved: {rfi_out}')
print(f'  Saved: {SAVE_DIR}rfi_baseline_{rfi_ts}.npy')
print('\n✅ PASS — RFI baseline spectrum saved  (share with Jordan and Phil)')


## Cell 5c — QICK dual spectrum: 0–200 MHz vs 0–276 MHz
Jordan's request: plot the full decimated band alongside the sweep band zoom
to check if any peaks align and compare resolution with the old rfsoc_sam spectrum.
**Uses the saved RFI baseline — no new board capture needed.**
Run Cell 5b first to generate `rfi_spec` and `freq_axis_mhz`.

In [ ]:
# ── Dual QICK spectrum plot ───────────────────────────────────────────────────
# Uses rfi_spec from Cell 5b (500-frame average, DAC off)
# Left panel:  0-276 MHz — full decimated band
# Right panel: 0-200 MHz — sweep band zoom, better frequency detail visible

if 'rfi_spec' not in dir():
    raise RuntimeError('Run Cell 5b first to capture rfi_spec')

rfi_mean = float(np.mean(rfi_spec))
rfi_rel  = rfi_spec - rfi_mean   # relative to mean floor

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
fig.suptitle(
    f'QICK direct sampling — RFI baseline spectra (DAC off, {N_BASELINE_FRAMES} frames)\n'
    f'fs_dec = {FS_DECIMATED_MHZ:.3f} MHz  |  {DF_KHZ:.2f} kHz/bin  |  Hann window',
    fontsize=11
)

# ── Left: full decimated band 0-276 MHz ──────────────────────────────────────
ax1.plot(freq_axis_mhz, rfi_rel, color='#00e5ff', lw=0.7, alpha=0.9)
ax1.axhline(0, color='grey', lw=0.6, linestyle='--', alpha=0.7, label='Mean floor')
ax1.axvspan(50, 200, alpha=0.08, color='cyan', label='50-200 MHz sweep band')
ax1.set_xlim(0, NYQUIST_DEC_MHZ)
ax1.set_ylim(-18, 22)
ax1.set_xlabel(f'Frequency (MHz)  [{DF_KHZ:.2f} kHz/bin]', fontsize=10)
ax1.set_ylabel('Power relative to mean (dB)', fontsize=10)
ax1.set_title(f'Full decimated band: 0 – {NYQUIST_DEC_MHZ:.1f} MHz', fontsize=10)
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.2)
ax1.annotate('Decimation\nfilter roll-in', xy=(15, 10), fontsize=7,
             color='#aaaaff', ha='center')
ax1.annotate('Decimation\nfilter roll-off', xy=(255, -8), fontsize=7,
             color='#aaaaff', ha='center')

# ── Right: sweep band zoom 0-200 MHz ────────────────────────────────────────
mask = freq_axis_mhz <= 205.0
ax2.plot(freq_axis_mhz[mask], rfi_rel[mask], color='#00e5ff', lw=0.9, alpha=0.9)
ax2.axhline(0, color='grey', lw=0.6, linestyle='--', alpha=0.7)
ax2.axvspan(50, 200, alpha=0.08, color='cyan')
ax2.set_xlim(0, 205)
ax2.set_xlabel(f'Frequency (MHz)  [{DF_KHZ:.2f} kHz/bin]', fontsize=10)
ax2.set_title('Sweep band zoom: 0 – 205 MHz', fontsize=10)
ax2.grid(True, alpha=0.2)

# Annotate any peaks > 3 dB above mean in the 50-200 MHz range
band_mask = (freq_axis_mhz >= 50) & (freq_axis_mhz <= 200)
peak_thresh = 6.0  # raised from 3 dB — avoids annotating noise variation
peaks_f = freq_axis_mhz[band_mask][rfi_rel[band_mask] > peak_thresh]
peaks_v = rfi_rel[band_mask][rfi_rel[band_mask] > peak_thresh]
# Group into clusters and label the maximum of each cluster
if len(peaks_f) > 0:
    clusters, current = [], [0]
    for j in range(1, len(peaks_f)):
        if peaks_f[j] - peaks_f[j-1] < DF_MHZ * 3:
            current.append(j)
        else:
            clusters.append(current); current = [j]
    clusters.append(current)
    for cl in clusters:
        best = cl[np.argmax(peaks_v[cl])]
        ax2.annotate(f'{peaks_f[best]:.1f} MHz\n{peaks_v[best]:.1f} dB',
                     xy=(peaks_f[best], peaks_v[best]),
                     xytext=(peaks_f[best]+3, peaks_v[best]+1.5),
                     fontsize=6, color='yellow',
                     arrowprops=dict(arrowstyle='->', color='yellow', lw=0.8))

plt.tight_layout()
dual_ts  = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
dual_out = f'{SAVE_DIR}qick_dual_spectrum_{dual_ts}.png'
plt.savefig(dual_out, dpi=150, bbox_inches='tight')
plt.show()
print(f'\n  Saved: {dual_out}')
print('\n✅ PASS — dual spectrum plot saved  (share with Jordan)')


## Cell 6 — DAC tone control function
Sets the QICK signal generator to produce a CW tone at a given frequency.

In [ ]:
from qick.averager_program import AveragerProgram

# ── CW tone + ADC capture ─────────────────────────────────────────────────────
# Key design decisions (confirmed from board debugging):
#
# 1. pulse_length_us = 20 us
#    200 us exceeded the 16-bit gen register (max ~53 us at 1228.8 MHz gen clock).
#    20 us = 24,576 gen cycles — safely within range AND >> ADC window (1.8 us).
#
# 2. Warmup trigger REMOVED from initialize()
#    Placing trigger() in initialize() caused a readout timing conflict.
#    Instead: soft_avgs=5 means the program runs 5 reps.
#    Rep 1 may capture a transitional frame; reps 2-5 capture clean tone frames.
#    The 5-rep average still gives good SNR.
#
# 3. adc_trig_offset = 200 tProc cycles = 0.47 us
#    Gives the DAC output time to propagate through the loopback cable
#    before the ADC starts sampling.

class CWToneProgram(AveragerProgram):
    """
    Fires CW tone on DAC_GEN and captures on ADC_CH in the same tProc program.
    Pulse fires at t=0, ADC trigger fires at t=adc_trig_offset cycles.
    soft_avgs=5 averages 5 reps — the tone is active for all of them.
    """
    def initialize(self):
        cfg = self.cfg
        self.declare_gen(ch=cfg['gen_ch'], nqz=1)
        self.declare_readout(ch=cfg['ro_ch'], length=cfg['readout_length'],
                             freq=0, gen_ch=cfg['gen_ch'])
        freq_reg = self.freq2reg(cfg['freq_mhz'], gen_ch=cfg['gen_ch'])
        self.set_pulse_registers(
            ch      = cfg['gen_ch'],
            style   = 'const',
            freq    = freq_reg,
            phase   = 0,
            gain    = cfg['gain'],
            length  = self.us2cycles(cfg['pulse_length_us'], gen_ch=cfg['gen_ch']),
            mode    = 'periodic',
            stdysel = 'zero',
        )
        self.synci(200)

    def body(self):
        cfg = self.cfg
        self.pulse(ch=cfg['gen_ch'], t=0)
        self.trigger(adcs=[cfg['ro_ch']], pins=[0],
                     adc_trig_offset=cfg['adc_trig_offset'])
        self.wait_all()
        self.sync_all(self.us2cycles(cfg['relax_delay']))


_TONE_CFG_BASE = {
    'gen_ch'         : DAC_GEN,
    'ro_ch'          : ADC_CH,
    'gain'           : int(DAC_AMPLITUDE * 32766),
    'readout_length' : READOUT_LENGTH,
    'adc_trig_offset': 200,    # 200 tProc cycles ≈ 0.47 us
    'pulse_length_us': 20.0,   # 20 us — safe for all gen clocks, >> ADC window
    'relax_delay'    : 1.0,    # 1 us between reps (was 1000 us — unnecessarily long)
    'soft_avgs'      : 5,      # 5 reps averaged — tone active for all reps
    'reps'           : 1,
}
_active_prog   = None
_pulses_loaded = False

def set_cw_tone(freq_mhz, amplitude=DAC_AMPLITUDE):
    """Prepare CWToneProgram. Call capture_tone_sample() next."""
    global _active_prog, _pulses_loaded
    cfg             = dict(_TONE_CFG_BASE)
    cfg['freq_mhz'] = freq_mhz
    cfg['gain']     = int(amplitude * 32766)
    _active_prog    = CWToneProgram(soc, cfg)
    _pulses_loaded  = False

def disable_tone():
    global _active_prog, _pulses_loaded
    _active_prog   = None
    _pulses_loaded = False

def capture_tone_sample():
    """
    Fire soft_avgs=5 tone+ADC reps and return averaged spectrum in dB.
    set_cw_tone(freq_mhz) must be called first.
    load_pulses=True only on first call per step — faster for subsequent frames.
    """
    global _pulses_loaded
    if _active_prog is None:
        raise RuntimeError('Call set_cw_tone(freq_mhz) before capture_tone_sample()')
    iq             = _active_prog.acquire_decimated(
                         soc, load_pulses=not _pulses_loaded, progress=False)
    _pulses_loaded = True
    i_data         = np.array(iq[0][0], dtype=np.float32)
    return compute_spectrum_db(i_data)


# ── Smoke test ────────────────────────────────────────────────────────────────
TEST_FREQ = 100.0
print(f'[SMOKE TEST] {TEST_FREQ} MHz — DAC_B → loopback cable → ADC_D')
print(f'  pulse_length_us : 20 us  (safe for all gen clocks, >> ADC window 1.8 us)')
print(f'  adc_trig_offset : 200 tProc cycles ≈ 0.47 us')
print(f'  soft_avgs       : 5 reps averaged per call')

set_cw_tone(TEST_FREQ)
test_spec = capture_tone_sample()
disable_tone()

eb  = np.argmin(np.abs(freq_axis_mhz - TEST_FREQ))
lo  = max(0, eb - 10)
hi  = min(len(test_spec), eb + 11)
ab  = lo + int(np.argmax(test_spec[lo:hi]))
am  = float(freq_axis_mhz[ab])
snr = float(test_spec[ab]) - float(np.median(test_spec))

print(f'\n  Expected: bin {eb} = {freq_axis_mhz[eb]:.2f} MHz')
print(f'  Measured: bin {ab} = {am:.2f} MHz  (error {am - TEST_FREQ:+.2f} MHz)')
print(f'  SNR: {snr:.1f} dB')

if snr > 5.0 and abs(am - TEST_FREQ) < 3.0:
    print(f'\n✅ PASS — tone visible, SNR = {snr:.1f} dB')
elif snr > 2.0:
    print(f'\n⚠️  MARGINAL — SNR = {snr:.1f} dB')
    print('  Try: set DAC_AMPLITUDE = 1.0 in Cell 4 and re-run from Cell 4.')
else:
    print('\n❌ FAIL — tone not visible (SNR < 2 dB)')
    print('  1. Check loopback cable DAC_B → ADC_D firmly connected')
    print('  2. Set DAC_AMPLITUDE = 1.0 in Cell 4 and re-run from Cell 4')
    print('  3. Re-run Cell 2 to reload the QICK overlay')
    print('  DO NOT proceed to Cell 7 until this passes.')


## Cell 7 — Baseline measurement (DAC off)
Measures the noise floor spectrum with no tone injected.  
This captures any residual RFI and the ADC thermal noise.  
**Unlike rfsoc_sam, there is no NCO spur at bin 1024 here.**  
**Disconnect any antenna — loopback cable only.**

In [ ]:
print(f'[BASELINE] Measuring noise floor ({N_BASELINE_FRAMES} frames, DAC off)...')
disable_tone()
time.sleep(0.5)

baseline_acc = np.zeros(len(freq_axis_mhz), dtype=np.float64)
for k in range(N_BASELINE_FRAMES):
    baseline_acc += capture_baseline_sample().astype(np.float64)
    if (k+1) % 100 == 0:
        print(f'  Frame {k+1}/{N_BASELINE_FRAMES}...')

baseline = (baseline_acc / N_BASELINE_FRAMES).astype(np.float32)

noise_mean = float(np.mean(baseline))
noise_std  = float(np.std(baseline))
noise_peak = float(np.max(baseline))
peak_mhz   = float(freq_axis_mhz[int(np.argmax(baseline))])

print(f'\n--- Baseline summary ---')
print(f'  Mean noise floor : {noise_mean:.2f} dB')
print(f'  Std dev          : {noise_std:.2f} dB')
print(f'  Strongest feature: {noise_peak:.2f} dB at {peak_mhz:.1f} MHz')

# QICK uses direct sampling — NCO spur at 1228.8 MHz cannot occur.
# Confirm the baseline looks flat (no dominant spurs).
if noise_std < 8.0 and (noise_peak - noise_mean) < 20.0:
    print('  ✅ No dominant spurs in baseline (QICK direct sampling confirmed)')
else:
    print(f'  ⚠️  Baseline has high variation (std={noise_std:.1f} dB) — check for RFI')

np.save(f'{SAVE_DIR}cw_baseline.npy', baseline)
print(f'\n  Saved: {SAVE_DIR}cw_baseline.npy')
print('\n PASS — baseline measured')

## Cell 8 — CW waterfall acquisition
Sweeps the DAC across the science band, collecting N_FRAMES_PER_STEP frames  
at each frequency. Stores the full baseline-subtracted waterfall array.  
**Jordan's pseudocode (Mar 2026):**
```
spectre = []
for i in range(n_steps):
    set_tone(f_i)
    for j in range(n_frames):
        s = acquire_spectrum
        spectre.append(s)
Spectra = np.array(spectre)
imshow(Spectra)
```

In [ ]:
N_TOTAL_FRAMES = N_STEPS * N_FRAMES_PER_STEP
print('=' * 60)
print('CW WATERFALL ACQUISITION')
print(f'  {N_STEPS} steps x {N_FRAMES_PER_STEP} frames = {N_TOTAL_FRAMES} rows')
print(f'  Sweep: {SWEEP_BAND_START:.3f} → {SWEEP_BAND_END:.3f} MHz')
print(f'  Tone formula (QICK): f_tone = f_DAC  (no F_LO offset)')
print('=' * 60)

# Allocate waterfall array
Spectra       = np.zeros((N_TOTAL_FRAMES, len(freq_axis_mhz)), dtype=np.float32)
dac_step_rows = []    # first row index for each DAC step
step_peak_snr = []    # peak SNR at each step (for gain curve)
row           = 0
t0            = time.time()

for step_idx, f_mhz in enumerate(DAC_FREQS_MHZ):

    set_cw_tone(f_mhz)   # warmup in initialize() handles settling
    dac_step_rows.append(row)

    step_acc = np.zeros(len(freq_axis_mhz), dtype=np.float64)

    for frame_idx in range(N_FRAMES_PER_STEP):
        raw_spec          = capture_tone_sample()
        subtracted        = raw_spec - baseline   # dB above noise floor
        Spectra[row]      = subtracted
        step_acc         += subtracted.astype(np.float64)
        row              += 1

    # Expected tone bin — use nearest bin in the correct decimated freq axis
    expected_bin = int(np.argmin(np.abs(freq_axis_mhz - f_mhz)))

    step_avg = (step_acc / N_FRAMES_PER_STEP).astype(np.float32)

    # Search ±5 bins around expected position
    lo = max(0, expected_bin - 5)
    hi = min(len(step_avg), expected_bin + 6)
    peak_snr = float(np.max(step_avg[lo:hi]))
    step_peak_snr.append(peak_snr)

    elapsed = time.time() - t0
    print(f'  Step {step_idx+1:3d}/{N_STEPS}  '
          f'DAC={f_mhz:.2f} MHz  '
          f'rows {dac_step_rows[-1]}–{row-1}  '
          f'tone_SNR={peak_snr:.1f} dB  '
          f'elapsed={elapsed:.1f}s')

disable_tone()
total_time = time.time() - t0
dt_ms      = total_time / N_TOTAL_FRAMES * 1000

print(f'\n  Done. {row} rows in {total_time:.1f}s  (ΔT = {dt_ms:.2f} ms/frame)')

# Save
ts = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
np.save(f'{SAVE_DIR}cw_waterfall_{ts}.npy',       Spectra)
np.save(f'{SAVE_DIR}cw_dac_step_rows_{ts}.npy',   np.array(dac_step_rows))
np.save(f'{SAVE_DIR}cw_dac_freqs_{ts}.npy',       DAC_FREQS_MHZ)
np.save(f'{SAVE_DIR}cw_freq_axis_{ts}.npy',       freq_axis_mhz)
np.save(f'{SAVE_DIR}cw_step_peak_snr_{ts}.npy',   np.array(step_peak_snr))
print(f'\n  Saved: cw_waterfall_{ts}.npy  (shape {Spectra.shape})')
print('\n PASS — waterfall acquired')

## Cell 9 — Waterfall plot
Full baseline-subtracted waterfall with expected tone positions marked.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 9),
                         gridspec_kw={'width_ratios': [3, 1]})

ax, ax2 = axes

# ── Left: waterfall imshow ────────────────────────────────────────────────────
n_rows  = Spectra.shape[0]
vmin    = -5.0
vmax    = float(np.nanpercentile(Spectra, 99.5))
vmax    = max(vmax, 10.0)

# Build frequency extent in MHz
f_min_mhz = float(freq_axis_mhz[0])
f_max_mhz = float(freq_axis_mhz[-1])

im = ax.imshow(
    Spectra,
    aspect='auto',
    extent=[f_min_mhz, f_max_mhz, n_rows, 0],
    vmin=vmin, vmax=vmax,
    cmap='viridis',
    interpolation='nearest',
    origin='upper'
)

# Mark step boundaries and expected tone positions
for i, r in enumerate(dac_step_rows):
    ax.axhline(r, color='white', lw=0.4, alpha=0.4, linestyle='--')
    tone_mhz = float(DAC_FREQS_MHZ[i])
    ax.axvline(tone_mhz, color='red', lw=0.8, alpha=0.6)
    if i % 5 == 0:
        ax.text(tone_mhz + max(SWEEP_SPAN * 0.01, DF_MHZ),
                r + N_FRAMES_PER_STEP * 0.5,
                f'{tone_mhz:.1f} MHz',
                fontsize=7, color='white', va='center',
                bbox=dict(facecolor='darkred', alpha=0.55, pad=1))

# Shade the science band
ax.axvspan(SWEEP_BAND_START, SWEEP_BAND_END, alpha=0.06, color='cyan')

ax.set_xlim(max(0, SWEEP_BAND_START - max(SWEEP_SPAN*0.05, 5)),
            SWEEP_BAND_END + max(SWEEP_SPAN*0.05, 5))
ax.set_xlabel(
    f'Frequency (MHz)  [{DF_KHZ:.2f} kHz/bin (true)  |  '
    f'tone at f_DAC directly (QICK direct sampling)]',
    fontsize=10
)
ax.set_ylabel('Frame index  (time,  each row = one get_frame())', fontsize=10)
ax.set_title(
    f'QICK CW Waterfall  (baseline subtracted)\n'
    f'Sweep: {SWEEP_BAND_START:.3f}–{SWEEP_BAND_END:.3f} MHz  |  '
    f'{N_FRAMES_PER_STEP} frames/step  |  {N_STEPS} steps  |  '
    f'colour = dB above noise floor',
    fontsize=10
)
plt.colorbar(im, ax=ax, label='dB above noise floor', shrink=0.8)

# ── Right: gain curve (peak tone power per step) ──────────────────────────────
steps = np.arange(N_STEPS)
ax2.barh(steps, step_peak_snr, color='steelblue', alpha=0.8, height=0.7)
ax2.set_yticks(steps)
ax2.set_yticklabels([f'{f:.1f}' for f in DAC_FREQS_MHZ], fontsize=7)
ax2.set_xlabel('Peak tone SNR\n(dB above noise floor)', fontsize=9)
ax2.set_ylabel('DAC frequency (MHz)', fontsize=9)
ax2.set_title('Preliminary\ngain curve g(ν)', fontsize=10)
ax2.axvline(0, color='grey', lw=0.8, linestyle=':')
ax2.grid(True, alpha=0.3, axis='x')
ax2.invert_yaxis()

plt.suptitle(
    f'RHINO RFSoC 4x2 — CW Calibration  (QICK overlay)\n'
    f'Science band: {SWEEP_BAND_START:.3f}–{SWEEP_BAND_END:.3f} MHz  |  '
    f'{ts}',
    fontsize=11
)
plt.tight_layout()
out = f'{SAVE_DIR}cw_waterfall_plot_{ts}.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'\n  Saved: {out}')
print('\n PASS — waterfall plot generated')

## Cell 10 — Averaged spectrum per step
Averages all N_FRAMES_PER_STEP frames at each DAC step.  
Produces a clean 27-row image showing the tone at centre of each row.  
This is the publishable result for the RASTI paper.

In [ ]:
# Define frequency extent locally in case Cell 9 (waterfall plot) was skipped
f_min_mhz = float(freq_axis_mhz[0])
f_max_mhz = float(freq_axis_mhz[-1])

# Build N_STEPS-row averaged image (one row per DAC step)
n_freq   = len(freq_axis_mhz)
avg_img  = np.zeros((N_STEPS, n_freq), dtype=np.float32)

for i, r_start in enumerate(dac_step_rows):
    r_end      = r_start + N_FRAMES_PER_STEP
    block      = Spectra[r_start:r_end]
    avg_img[i] = np.mean(block, axis=0)

vmin_avg = -5.0
vmax_avg = float(np.nanpercentile(avg_img, 99))
vmax_avg = max(vmax_avg, 10.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 8),
                         gridspec_kw={'width_ratios': [3, 1]})
ax, ax2 = axes

# ── Left: clean averaged waterfall ───────────────────────────────────────────
img_extent = [f_min_mhz, f_max_mhz, N_STEPS - 0.5, -0.5]
im = ax.imshow(
    avg_img,
    aspect='auto',
    extent=img_extent,
    vmin=vmin_avg, vmax=vmax_avg,
    cmap='viridis',
    interpolation='nearest',
    origin='upper'
)

# Mark expected tone position as red dashed line at the actual DAC frequency
for i, f_mhz in enumerate(DAC_FREQS_MHZ):
    ax.axvline(f_mhz, color='red', lw=0.6, alpha=0.5)

ax.set_yticks(range(N_STEPS))
ax.set_yticklabels([f'{f:.1f} MHz' for f in DAC_FREQS_MHZ], fontsize=7)
ax.set_xlim(max(0, SWEEP_BAND_START - max(SWEEP_SPAN*0.05, 5)),
            SWEEP_BAND_END + max(SWEEP_SPAN*0.05, 5))
ax.set_xlabel(
    f'Frequency (MHz)  [{DF_KHZ:.2f} kHz/bin (true)  |  '
    f'tone = f_DAC  (QICK direct sampling, no offset)]',
    fontsize=10
)
ax.set_ylabel('DAC frequency step', fontsize=10)
ax.set_title(
    f'Averaged CW Waterfall — {N_FRAMES_PER_STEP} frames/step\n'
    f'Red lines = expected tone position  |  colour = dB above noise floor',
    fontsize=10
)
plt.colorbar(im, ax=ax, label='dB above noise floor', shrink=0.8)

# ── Right: gain curve ────────────────────────────────────────────────────────
# Peak power in ±3-bin window around expected tone bin
gain_curve = np.zeros(N_STEPS, dtype=np.float32)
for i, f_mhz in enumerate(DAC_FREQS_MHZ):
    expected_bin = int(np.argmin(np.abs(freq_axis_mhz - f_mhz)))
    lo = max(0, expected_bin - 3)
    hi = min(n_freq, expected_bin + 4)
    gain_curve[i] = float(np.max(avg_img[i, lo:hi]))
    if spur_mask[i]:
        gain_curve[i] = float('nan')   # flag corrupted channel

steps = np.arange(N_STEPS)
bar_colors = ['#c0392b' if spur_mask[i] else 'steelblue' for i in range(N_STEPS)]
ax2.barh(steps, gain_curve, color=bar_colors, alpha=0.85, height=0.7)
from matplotlib.patches import Patch
ax2.legend(handles=[
    Patch(color='steelblue', label='Valid channel'),
    Patch(color='#c0392b',   label='Flagged (board spur)'),
], fontsize=8)
ax2.set_yticks(steps)
ax2.set_yticklabels([f'{f:.1f}' for f in DAC_FREQS_MHZ], fontsize=7)
ax2.set_xlabel('Tone power g(ν)\n(dB above noise floor)', fontsize=9)
ax2.set_title('Gain curve g(ν)\n(peak in ±3-bin window)', fontsize=10)
ax2.axvline(0, color='grey', lw=0.8, linestyle=':')
ax2.grid(True, alpha=0.3, axis='x')
ax2.invert_yaxis()

plt.suptitle(
    f'RHINO RFSoC 4x2 — CW Calibration Gain Curve  (QICK)\n'
    f'Sweep {SWEEP_BAND_START:.3f}–{SWEEP_BAND_END:.3f} MHz  |  '
    f'{N_STEPS} steps  |  {N_FRAMES_PER_STEP} frames averaged per step',
    fontsize=11
)
plt.tight_layout()
out2 = f'{SAVE_DIR}cw_gain_curve_{ts}.png'
plt.savefig(out2, dpi=150, bbox_inches='tight')
plt.show()

# Also save the gain curve as a numpy array for the RASTI paper
np.save(f'{SAVE_DIR}cw_gain_curve_{ts}.npy',    gain_curve)
np.save(f'{SAVE_DIR}cw_avg_waterfall_{ts}.npy', avg_img)
print(f'\n  Saved: {out2}')
print(f'  Saved: {SAVE_DIR}cw_gain_curve_{ts}.npy')
print('\n PASS — averaged waterfall and gain curve generated')

## Cell 10b — Channel response: three windows compared
Shows the single-tone channel response (centre bin bright, neighbours dropping)
under Rectangular (no window), Hann, and Blackman.
Jordan's request: see the leakage into neighbouring channels for each window.

**Run AFTER Cell 8 (Spectra and avg_img must exist).**
Captures fresh raw ADC samples at the best-SNR frequency, applies all three
windows to the same data, and plots the results side by side.

In [ ]:
# ── Channel response: three windows on the same raw data ─────────────────────
# Captures baseline frames FIRST (DAC off), then tone frames (DAC on).
# Applies Rectangular, Hann, and Blackman to the same raw data.
# Produces three side-by-side panels showing leakage under each window.

WIN_RESP_FREQ_MHZ = None    # None = auto-pick best SNR step; or set e.g. 150.0
WIN_RESP_FRAMES   = 200
WIN_RESP_BINS     = 12      # ±12 bins around tone centre
WIN_RESP_RL       = 993   # 993+2=995 (odd) — DMA delivers 995 cleanly
SPUR_LO_MHZ       = 93.0
SPUR_HI_MHZ       = 111.0

if 'avg_img' not in dir():
    raise RuntimeError('Run Cells 8 and 10 first')

# Pick frequency
if WIN_RESP_FREQ_MHZ is None:
    null_mask   = (DAC_FREQS_MHZ < SPUR_LO_MHZ) | (DAC_FREQS_MHZ > SPUR_HI_MHZ)
    valid_steps = np.where(null_mask)[0]
    step_snr    = np.zeros(N_STEPS)
    for _i, _f in enumerate(DAC_FREQS_MHZ):
        _eb = int(np.argmin(np.abs(freq_axis_mhz - _f)))
        _lo, _hi = max(0, _eb-3), min(len(avg_img[_i]), _eb+4)
        step_snr[_i] = float(np.max(avg_img[_i, _lo:_hi]))
    best_step = valid_steps[np.argmax(step_snr[valid_steps])]
    f_resp = float(DAC_FREQS_MHZ[best_step])
else:
    f_resp = float(WIN_RESP_FREQ_MHZ)

eb_resp = int(np.argmin(np.abs(freq_axis_mhz - f_resp)))
print(f'[CHANNEL RESPONSE] Tone at {f_resp:.4f} MHz  (bin {eb_resp})')

# ── Step 1: Capture baseline frames FIRST (DAC off) ───────────────────────────
print(f'  Step 1: capturing {WIN_RESP_FRAMES} baseline frames (DAC off)...')
_resp_no_cfg = {'ro_ch': ADC_CH, 'readout_length': WIN_RESP_RL,
                'adc_trig_offset': 200, 'soft_avgs': 1, 'reps': 1,
                'relax_delay': 1.0}
_resp_no_prog = NoToneProgram(soc, _resp_no_cfg)

baseline_frames = []
for _ in range(WIN_RESP_FRAMES):
    iq = _resp_no_prog.acquire_decimated(soc, load_pulses=False, progress=False)
    i_data = np.array(iq[0][0], dtype=np.float32)
    if len(i_data) < WIN_RESP_RL:
        i_data = np.pad(i_data, (0, WIN_RESP_RL - len(i_data)))
    baseline_frames.append(i_data[:WIN_RESP_RL])
baseline_frames = np.array(baseline_frames, dtype=np.float32)
print(f'  Baseline captured: shape={baseline_frames.shape}')

# ── Step 2: Capture tone frames (DAC on) ──────────────────────────────────────
print(f'  Step 2: capturing {WIN_RESP_FRAMES} tone frames at {f_resp:.4f} MHz...')
_resp_cfg             = dict(_TONE_CFG_BASE)
_resp_cfg['readout_length'] = WIN_RESP_RL
_resp_cfg['freq_mhz']       = f_resp
_resp_prog            = CWToneProgram(soc, _resp_cfg)

raw_frames = []
for k in range(WIN_RESP_FRAMES):
    iq = _resp_prog.acquire_decimated(
        soc, load_pulses=(k == 0), progress=False)
    i_data = np.array(iq[0][0], dtype=np.float32)
    if len(i_data) < WIN_RESP_RL:
        i_data = np.pad(i_data, (0, WIN_RESP_RL - len(i_data)))
    raw_frames.append(i_data[:WIN_RESP_RL])
disable_tone()
raw_frames = np.array(raw_frames, dtype=np.float32)
print(f'  Tone frames captured: shape={raw_frames.shape}')

# ── Step 3: Apply each window and compute channel response ────────────────────
WINDOW_RESP_DEFS = {
    'Rectangular': {'w': np.ones(WIN_RESP_RL, dtype=np.float32),
                    'color': '#e74c3c', 'sl': -13.3},
    'Hann':        {'w': np.hanning(WIN_RESP_RL).astype(np.float32),
                    'color': '#2ecc71', 'sl': -31.5},
    'Blackman':    {'w': np.blackman(WIN_RESP_RL).astype(np.float32),
                    'color': '#3498db', 'sl': -58.1},
}
win_fax = np.fft.rfftfreq(WIN_RESP_RL, d=1.0/(FS_DECIMATED_MHZ*1e6)) / 1e6
eb_w    = int(np.argmin(np.abs(win_fax - f_resp)))

resp_data = {}
for wname, wdict in WINDOW_RESP_DEFS.items():
    w  = wdict['w']
    wp = float(np.sum(w**2))

    # Baseline spectrum
    bl_acc = np.zeros(WIN_RESP_RL // 2 + 1, dtype=np.float64)
    for frame in baseline_frames:
        bl_acc += np.abs(np.fft.rfft(frame * w))**2 / wp
    bl_spec = 10.0 * np.log10(np.maximum(bl_acc / WIN_RESP_FRAMES, 1e-30))

    # Tone spectrum
    tone_acc = np.zeros(WIN_RESP_RL // 2 + 1, dtype=np.float64)
    for frame in raw_frames:
        tone_acc += np.abs(np.fft.rfft(frame * w))**2 / wp
    tone_spec = 10.0 * np.log10(np.maximum(tone_acc / WIN_RESP_FRAMES, 1e-30))

    # Baseline-subtracted, normalised to peak = 0 dB
    subtracted = tone_spec - bl_spec
    lo = max(0, eb_w - WIN_RESP_BINS)
    hi = min(len(subtracted), eb_w + WIN_RESP_BINS + 1)
    y  = subtracted[lo:hi]
    y_norm = y - float(np.max(y))

    resp_data[wname] = {
        'y_norm': y_norm,
        'x_bins': np.arange(lo, hi) - eb_w,
        'n1': float(y_norm[WIN_RESP_BINS + 1]) if WIN_RESP_BINS+1 < len(y_norm) else float('nan'),
        'n2': float(y_norm[WIN_RESP_BINS + 2]) if WIN_RESP_BINS+2 < len(y_norm) else float('nan'),
        'snr': float(np.max(subtracted[max(0,eb_w-3):eb_w+4])),
    }
    print(f'  {wname:<14}: peak SNR={resp_data[wname]["snr"]:.1f} dB  '
          f'bin+1={resp_data[wname]["n1"]:.1f} dB  '
          f'bin+2={resp_data[wname]["n2"]:.1f} dB')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 6), sharey=True)

for ax, (wname, wdict) in zip(axes, WINDOW_RESP_DEFS.items()):
    d     = resp_data[wname]
    color = wdict['color']
    ax.bar(d['x_bins'], d['y_norm'], color=color, alpha=0.75, width=0.75)
    ax.axvline(0, color='white', lw=1.2, linestyle='--', alpha=0.8,
               label='Tone centre')
    ax.axhline(wdict['sl'], color='orange', lw=1.0, linestyle=':',
               label=f'1st sidelobe ({wdict["sl"]:.1f} dB)')
    ax.axvline(-0.5, color='yellow', lw=0.8, linestyle='--', alpha=0.5,
               label='Channel boundary')
    ax.axvline( 0.5, color='yellow', lw=0.8, linestyle='--', alpha=0.5)
    ax.set_xlim(-WIN_RESP_BINS - 0.5, WIN_RESP_BINS + 0.5)
    ax.set_ylim(-90, 5)
    ax.set_xlabel('Relative bin index', fontsize=9)
    ax.set_title(f'{wname}\n1st sidelobe: {wdict["sl"]:.1f} dB',
                 fontsize=10, color=color, fontweight='bold')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.2)
    ax.text(0.02, 0.04,
            f'SNR: {d["snr"]:.1f} dB\nbin+1: {d["n1"]:.1f} dB\nbin+2: {d["n2"]:.1f} dB',
            transform=ax.transAxes, fontsize=8, color='white',
            bbox=dict(facecolor='#333', alpha=0.7, pad=3))

axes[0].set_ylabel('Power relative to peak (dB)', fontsize=10)
fig.suptitle(
    f'RHINO RFSoC 4x2 — Channel response: window comparison\n'
    f'Tone at {f_resp:.4f} MHz  |  {WIN_RESP_FRAMES} frames  |  '
    f'channel width = {DF_KHZ:.2f} kHz',
    fontsize=11
)
plt.tight_layout()
cr_out = (f'{SAVE_DIR}channel_response_windows_'
          f'{datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")}.png')
plt.savefig(cr_out, dpi=150, bbox_inches='tight')
plt.show()
print(f'\n  Saved: {cr_out}')
print()
print('--- Summary ---')
print(f'  {"Window":<14}  {"SNR (dB)":>10}  {"bin+1 (dB)":>12}  '
      f'{"bin+2 (dB)":>12}  {"vs theory":>12}')
print('  ' + '-'*64)
for wname, wdict in WINDOW_RESP_DEFS.items():
    d   = resp_data[wname]
    dev = d['n1'] - wdict['sl']
    print(f'  {wname:<14}  {d["snr"]:>10.1f}  {d["n1"]:>12.1f}  '
          f'{d["n2"]:>12.1f}  {dev:>+11.1f} dB vs theory')
print()
print('  Rectangular: worst leakage (neighbours closest to 0 dB)')
print('  Hann:        moderate leakage (-31.5 dB theory)')
print('  Blackman:    best suppression (-58.1 dB theory)')
print()
print('✅ PASS — channel response window comparison complete')


## Cell 10c — Window comparison: full sweep with Rectangular, Hann, and Blackman
Jordan's request: run the complete 50–200 MHz CW sweep three times, once per window,
and compare how the gain curve g(ν) changes across the band under each window.

**Runs the full acquisition three times — takes ~3× the normal sweep time (~12 minutes).**
Run AFTER Cells 4–7 (config, capture functions, baseline all defined).
Cell 8 does NOT need to have run first — this cell runs its own sweeps.

In [ ]:
# ── Full sweep window comparison ─────────────────────────────────────────────
# Runs the complete DAC frequency sweep three times, once per window function.
# For each window: measures the gain curve g(ν) across the full band.
# Produces a combined plot showing g(ν) for all three windows on one axis.

WIN_COMP_FRAMES = 200   # frames per step per window (same as main sweep)

# QICK DMA fix: READOUT_LENGTH=1000 causes 'Requested 1002 but got 1001'
# because QICK pads by +2 for IQ framing. Use 996 → QICK requests 998 ✓
WIN_READOUT_LENGTH = 993   # 993+2=995 (odd) — DMA delivers 995 cleanly
_TONE_CFG_BASE['readout_length'] = WIN_READOUT_LENGTH
_no_tone_prog = NoToneProgram(soc, {
    'ro_ch': ADC_CH, 'readout_length': WIN_READOUT_LENGTH,
    'adc_trig_offset': 200, 'soft_avgs': 1, 'reps': 1, 'relax_delay': 1.0,
})
win_freq_axis = np.fft.rfftfreq(WIN_READOUT_LENGTH,
                                 d=1.0/(FS_DECIMATED_MHZ*1e6)) / 1e6
print(f'[DMA FIX] Using readout_length={WIN_READOUT_LENGTH} for window sweep')
print(f'  Freq axis: {len(win_freq_axis)} bins, 0 to {win_freq_axis[-1]:.1f} MHz')

WINDOW_SWEEP_DEFS = {
    'Rectangular': {
        'w':        np.ones(WIN_READOUT_LENGTH, dtype=np.float32),
        'color':    '#e74c3c',
        'ls':       '-',
        'sidelobe': -13.3,
    },
    'Hann': {
        'w':        np.hanning(WIN_READOUT_LENGTH).astype(np.float32),
        'color':    '#2ecc71',
        'ls':       '--',
        'sidelobe': -31.5,
    },
    'Blackman': {
        'w':        np.blackman(WIN_READOUT_LENGTH).astype(np.float32),
        'color':    '#3498db',
        'ls':       ':',
        'sidelobe': -58.1,
    },
}

win_gain_curves = {}   # gain_curve_dB[window_name] = array of length N_STEPS

for wname, wdict in WINDOW_SWEEP_DEFS.items():
    w  = wdict['w'][:WIN_READOUT_LENGTH].astype(np.float32)
    wp = float(np.sum(w**2))
    print(f'\n[WINDOW SWEEP] {wname}  (sidelobe {wdict["sidelobe"]:.1f} dB)')
    print(f'  {N_STEPS} steps x {WIN_COMP_FRAMES} frames each...')

    # Recompute baseline spectrum with this window
    print('  Computing baseline with this window...')
    baseline_acc_w = np.zeros(WIN_READOUT_LENGTH // 2 + 1, dtype=np.float64)
    for _ in range(N_BASELINE_FRAMES):
        iq = _no_tone_prog.acquire_decimated(soc, load_pulses=False, progress=False)
        i_data = np.array(iq[0][0], dtype=np.float32)
        if len(i_data) < WIN_READOUT_LENGTH:
            i_data = np.pad(i_data, (0, WIN_READOUT_LENGTH - len(i_data)))
        windowed = i_data[:WIN_READOUT_LENGTH] * w
        power    = np.abs(np.fft.rfft(windowed))**2 / wp
        baseline_acc_w += power
    baseline_w = 10.0 * np.log10(
        np.maximum(baseline_acc_w / N_BASELINE_FRAMES, 1e-30)
    ).astype(np.float32)

    # Run sweep
    gain_curve_w = np.zeros(N_STEPS, dtype=np.float32)
    t0 = time.time()

    for step_idx, f_mhz in enumerate(DAC_FREQS_MHZ):
        set_cw_tone(f_mhz)
        step_acc = np.zeros(WIN_READOUT_LENGTH // 2 + 1, dtype=np.float64)

        for frame_idx in range(WIN_COMP_FRAMES):
            iq = _active_prog.acquire_decimated(
                soc, load_pulses=(frame_idx == 0), progress=False)
            i_data = np.array(iq[0][0], dtype=np.float32)
            if len(i_data) < WIN_READOUT_LENGTH:
                i_data = np.pad(i_data, (0, WIN_READOUT_LENGTH - len(i_data)))
            windowed = i_data[:WIN_READOUT_LENGTH] * w
            power    = np.abs(np.fft.rfft(windowed))**2 / wp
            step_acc += power

        disable_tone()
        avg_spec = 10.0 * np.log10(
            np.maximum(step_acc / WIN_COMP_FRAMES, 1e-30)
        ).astype(np.float32)
        subtracted = avg_spec - baseline_w

        eb = int(np.argmin(np.abs(win_freq_axis - f_mhz)))
        lo = max(0, eb - 3)
        hi = min(len(subtracted), eb + 4)
        gain_curve_w[step_idx] = float(np.max(subtracted[lo:hi]))

        elapsed = time.time() - t0
        print(f'  Step {step_idx+1:3d}/{N_STEPS}  {f_mhz:.2f} MHz  '
              f'g={gain_curve_w[step_idx]:.1f} dB  elapsed={elapsed:.0f}s')

    win_gain_curves[wname] = gain_curve_w
    np.save(
        f'{SAVE_DIR}gain_curve_{wname.lower()}_{datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")}.npy',
        gain_curve_w
    )
    print(f'  Done. Mean g(v) = {float(np.mean(gain_curve_w)):.2f} dB')

# ── Combined plot ─────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Top panel: gain curves overlaid
for wname, wdict in WINDOW_SWEEP_DEFS.items():
    gc = win_gain_curves[wname]
    ax1.plot(DAC_FREQS_MHZ, gc,
             color=wdict['color'], lw=2.0, linestyle=wdict['ls'],
             marker='o', markersize=4,
             label=f'{wname}  (sidelobe {wdict["sidelobe"]:.1f} dB)')

ax1.axvspan(93, 111, alpha=0.12, color='red', label='Board spur region (93-111 MHz)')
ax1.set_xlim(SWEEP_BAND_START - 5, SWEEP_BAND_END + 5)
ax1.set_xlabel('DAC frequency (MHz)', fontsize=10)
ax1.set_ylabel('Peak tone SNR (dB above noise floor)', fontsize=10)
ax1.set_title('Gain curve g(v) comparison — three window functions', fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.25)

# Bottom panel: difference from Hann (reference)
hann_gc = win_gain_curves['Hann']
for wname, wdict in WINDOW_SWEEP_DEFS.items():
    if wname == 'Hann':
        continue
    diff = win_gain_curves[wname] - hann_gc
    ax2.plot(DAC_FREQS_MHZ, diff,
             color=wdict['color'], lw=2.0, linestyle=wdict['ls'],
             marker='o', markersize=4,
             label=f'{wname} minus Hann')

ax2.axhline(0, color='#2ecc71', lw=1.0, linestyle='--', alpha=0.7, label='Hann (reference)')
ax2.axvspan(93, 111, alpha=0.12, color='red')
ax2.set_xlim(SWEEP_BAND_START - 5, SWEEP_BAND_END + 5)
ax2.set_xlabel('DAC frequency (MHz)', fontsize=10)
ax2.set_ylabel('Difference from Hann (dB)', fontsize=10)
ax2.set_title('Window effect on g(v) — difference from Hann reference', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.25)

ts_wc = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
fig.suptitle(
    f'RHINO RFSoC 4x2 — Window comparison: full sweep  |  '
    f'{SWEEP_BAND_START:.1f}-{SWEEP_BAND_END:.1f} MHz  |  '
    f'{N_STEPS} steps  |  {WIN_COMP_FRAMES} frames/step',
    fontsize=11
)
plt.tight_layout()
wc_out = f'{SAVE_DIR}window_sweep_comparison_{ts_wc}.png'
plt.savefig(wc_out, dpi=150, bbox_inches='tight')
plt.show()

print(f'\n  Saved: {wc_out}')
print()
print('--- Window sweep summary ---')
print(f'  {"Window":<14}  {"Mean g(v) (dB)":>16}  {"Std dev (dB)":>14}')
print('  ' + '-'*48)
for wname in WINDOW_SWEEP_DEFS:
    gc = win_gain_curves[wname]
    print(f'  {wname:<14}  {float(np.mean(gc)):>16.2f}  {float(np.std(gc)):>14.2f}')
print()
print('  If g(v) curves are similar across all three windows:')
print('    -> Hann is fine as the default (good leakage suppression, standard)')
print('  If Rectangular shows significantly higher variance:')
print('    -> Spectral leakage is affecting the gain curve measurements')
print('    -> Blackman or Hann recommended for the calibration')
print()
# Restore original readout_length for subsequent cells
_TONE_CFG_BASE['readout_length'] = READOUT_LENGTH
_no_tone_prog = NoToneProgram(soc, {
    'ro_ch': ADC_CH, 'readout_length': READOUT_LENGTH,
    'adc_trig_offset': 200, 'soft_avgs': 1, 'reps': 1, 'relax_delay': 1.0,
})
print('  Restored readout_length to', READOUT_LENGTH, 'for subsequent cells')
print('✅ PASS — window sweep comparison complete')


## Cell 10d — Fine-step window response measurement
Jordan's request: sweep the DAC in fine steps across 5 channel widths,
measuring how much power appears in the **fixed** target channel as the tone moves.

This gives the measured window frequency response function — showing exactly how
quickly energy leaks from an off-centre tone into the target channel under each window.

**Run AFTER Cells 5 and 6. Cell 8 does NOT need to have run first.**

In [ ]:
# ── Fine-step window response measurement ────────────────────────────────────
# Sweeps the DAC in fine steps across FINE_N_CHANNELS channel widths.
# At each DAC step: measures power in the FIXED target channel.
# Result: power vs offset = measured window response function.
# Repeated for Rectangular, Hann, Blackman — theoretical curves overlaid.

FINE_CENTRE_MHZ = 150.1    # target channel centre — away from board spur
FINE_N_CHANNELS = 5        # span in channels
FINE_N_STEPS    = 64       # DAC steps across span (power of 2 recommended)
FINE_FRAMES     = 100      # ADC frames per DAC step
FINE_RL         = 993   # 993+2=995 (odd) — DMA delivers 995 cleanly

# Derived parameters
FINE_DF_MHZ  = FS_DECIMATED_MHZ / FINE_RL
FINE_SPAN    = FINE_N_CHANNELS * FINE_DF_MHZ
FINE_START   = FINE_CENTRE_MHZ - FINE_SPAN / 2
FINE_STOP    = FINE_CENTRE_MHZ + FINE_SPAN / 2
FINE_FREQS   = np.linspace(FINE_START, FINE_STOP, FINE_N_STEPS)
FINE_OFFSETS = (FINE_FREQS - FINE_CENTRE_MHZ) / FINE_DF_MHZ

fine_fax    = np.fft.rfftfreq(FINE_RL, d=1.0/(FS_DECIMATED_MHZ*1e6)) / 1e6
TARGET_BIN  = int(np.argmin(np.abs(fine_fax - FINE_CENTRE_MHZ)))
TARGET_MHZ  = float(fine_fax[TARGET_BIN])

print('--- Fine-step window response ---')
print(f'  Centre     : {FINE_CENTRE_MHZ} MHz -> bin {TARGET_BIN} = {TARGET_MHZ:.4f} MHz')
print(f'  Chan width : {FINE_DF_MHZ*1e3:.3f} kHz')
print(f'  Span       : {FINE_N_CHANNELS} channels = {FINE_SPAN*1e3:.2f} kHz')
print(f'  Range      : {FINE_START:.4f} to {FINE_STOP:.4f} MHz')
print(f'  Steps      : {FINE_N_STEPS}  ({(FINE_FREQS[1]-FINE_FREQS[0])*1e3:.2f} kHz/step)')
print()

FINE_WIN_DEFS = {
    'Rectangular': {'w': np.ones(FINE_RL,  dtype=np.float32),
                    'color': '#e74c3c', 'ls': '-',  'sl': -13.3},
    'Hann':        {'w': np.hanning(FINE_RL).astype(np.float32),
                    'color': '#2ecc71', 'ls': '--', 'sl': -31.5},
    'Blackman':    {'w': np.blackman(FINE_RL).astype(np.float32),
                    'color': '#3498db', 'ls': ':',  'sl': -58.1},
}

_fine_no_cfg  = {'ro_ch': ADC_CH, 'readout_length': FINE_RL,
                 'adc_trig_offset': 200, 'soft_avgs': 1,
                 'reps': 1, 'relax_delay': 1.0}
_fine_no_prog = NoToneProgram(soc, _fine_no_cfg)

# Step 1: capture baseline frames (DAC off)
print('[STEP 1] Capturing 500 baseline frames (DAC off)...')
fine_bl_frames = []
for _ in range(500):
    iq    = _fine_no_prog.acquire_decimated(soc, load_pulses=False, progress=False)
    i_raw = np.array(iq[0][0], dtype=np.float32)
    if len(i_raw) < FINE_RL:
        i_raw = np.pad(i_raw, (0, FINE_RL - len(i_raw)))
    fine_bl_frames.append(i_raw[:FINE_RL])
fine_bl_frames = np.array(fine_bl_frames, dtype=np.float32)

# Per-window baseline power at target bin
fine_bl_power = {}
for wname, wdict in FINE_WIN_DEFS.items():
    w  = wdict['w']
    wp = float(np.sum(w**2))
    acc = 0.0
    for frame in fine_bl_frames:
        acc += (np.abs(np.fft.rfft(frame * w))**2 / wp)[TARGET_BIN]
    fine_bl_power[wname] = acc / len(fine_bl_frames)

# Step 2: fine-step sweep
print('[STEP 2] Running fine-step sweep...')
fine_response = {wname: np.zeros(FINE_N_STEPS) for wname in FINE_WIN_DEFS}
_fine_cfg_base = dict(_TONE_CFG_BASE)
_fine_cfg_base['readout_length'] = FINE_RL

for step_idx, f_mhz in enumerate(FINE_FREQS):
    cfg             = dict(_fine_cfg_base)
    cfg['freq_mhz'] = f_mhz
    prog            = CWToneProgram(soc, cfg)
    frames = []
    for k in range(FINE_FRAMES):
        iq    = prog.acquire_decimated(soc, load_pulses=(k==0), progress=False)
        i_raw = np.array(iq[0][0], dtype=np.float32)
        if len(i_raw) < FINE_RL:
            i_raw = np.pad(i_raw, (0, FINE_RL - len(i_raw)))
        frames.append(i_raw[:FINE_RL])
    disable_tone()
    frames = np.array(frames, dtype=np.float32)

    for wname, wdict in FINE_WIN_DEFS.items():
        w  = wdict['w']
        wp = float(np.sum(w**2))
        acc = 0.0
        for frame in frames:
            acc += (np.abs(np.fft.rfft(frame * w))**2 / wp)[TARGET_BIN]
        signal = max(acc / FINE_FRAMES - fine_bl_power[wname], 1e-30)
        fine_response[wname][step_idx] = 10.0 * math.log10(signal)

    if (step_idx + 1) % 8 == 0 or step_idx == 0:
        print(f'  Step {step_idx+1:3d}/{FINE_N_STEPS}  {f_mhz:.4f} MHz  '
              f'offset={FINE_OFFSETS[step_idx]:+.3f} ch  '
              f'Hann={fine_response["Hann"][step_idx]:.1f} dB')

# Normalise to peak
for wname in FINE_WIN_DEFS:
    fine_response[wname] -= float(np.max(fine_response[wname]))

# Theoretical response function
pad_th    = 65536
off_th    = np.linspace(-FINE_N_CHANNELS/2, FINE_N_CHANNELS/2, 2000)

def theoretical_response_dB(w_arr, offsets_ch, pad=65536):
    W    = np.fft.rfft(w_arr, n=pad)
    W_dB = 20 * np.log10(np.abs(W) / np.max(np.abs(W)) + 1e-15)
    idxs = np.clip((np.abs(offsets_ch) * pad / len(w_arr)).astype(int),
                   0, len(W_dB)-1)
    return W_dB[idxs]

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=False)

for wname, wdict in FINE_WIN_DEFS.items():
    col = wdict['color']
    ls  = wdict['ls']
    r   = fine_response[wname]
    ax1.plot(FINE_OFFSETS, r, color=col, lw=2.0, linestyle=ls,
             marker='o', markersize=3, label=f'{wname} (measured)')
    th = theoretical_response_dB(wdict['w'], off_th)
    ax1.plot(off_th, th, color=col, lw=0.7, linestyle=ls, alpha=0.35)

for v in [-2.5,-1.5,-0.5,0.5,1.5,2.5]:
    ax1.axvline(v, color='grey', lw=0.4, linestyle=':', alpha=0.3)
ax1.axvline(0, color='white', lw=1.0, linestyle='--', alpha=0.6,
            label='Tone centre (target bin)')
ax1.axvline(-0.5, color='yellow', lw=0.9, linestyle=':', alpha=0.6,
            label='Channel boundary')
ax1.axvline( 0.5, color='yellow', lw=0.9, linestyle=':', alpha=0.6)
ax1.set_ylim(-85, 5)
ax1.set_xlabel('Tone offset from target channel centre (channels)', fontsize=11)
ax1.set_ylabel('Power in target channel (dB, normalised)', fontsize=11)
ax1.set_title('Measured window response (bold) vs theoretical (faint)\n'
              'Power in target channel as DAC tone sweeps across 5 channels',
              fontsize=11)
ax1.legend(fontsize=9, ncol=2)
ax1.grid(True, alpha=0.25)
for ch in range(-2, 3):
    ax1.text(ch, -81, f'ch{ch:+d}', ha='center', fontsize=7,
             color='white', alpha=0.5)

for wname, wdict in FINE_WIN_DEFS.items():
    r   = fine_response[wname]
    th  = theoretical_response_dB(wdict['w'], FINE_OFFSETS)
    ax2.plot(FINE_OFFSETS, r - th, color=wdict['color'], lw=1.5,
             linestyle=wdict['ls'], marker='o', markersize=3,
             label=f'{wname}: measured minus theoretical')

ax2.axhline(0, color='grey', lw=1.0, linestyle='--', alpha=0.7)
ax2.axvline(-0.5, color='yellow', lw=0.9, linestyle=':', alpha=0.5)
ax2.axvline( 0.5, color='yellow', lw=0.9, linestyle=':', alpha=0.5)
ax2.set_xlabel('Tone offset from target channel centre (channels)', fontsize=11)
ax2.set_ylabel('Measured minus theoretical (dB)', fontsize=11)
ax2.set_title('Deviation from theoretical window response\n'
              'Near zero = window behaves as expected; '
              'deviations = freq2reg quantisation or noise effects',
              fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.25)

ts_fine = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
fig.suptitle(
    f'RHINO RFSoC 4x2 - Fine-step window response\n'
    f'Centre {FINE_CENTRE_MHZ} MHz | Span {FINE_N_CHANNELS} ch | '
    f'{FINE_N_STEPS} steps | {FINE_FRAMES} frames/step',
    fontsize=12
)
plt.tight_layout()
fine_out = f'{SAVE_DIR}fine_step_window_response_{ts_fine}.png'
plt.savefig(fine_out, dpi=150, bbox_inches='tight')
plt.show()

np.save(f'{SAVE_DIR}fine_step_offsets_{ts_fine}.npy',  FINE_OFFSETS)
np.save(f'{SAVE_DIR}fine_step_freqs_{ts_fine}.npy',    FINE_FREQS)
for wname in FINE_WIN_DEFS:
    np.save(f'{SAVE_DIR}fine_step_{wname.lower()}_{ts_fine}.npy',
            fine_response[wname])

_TONE_CFG_BASE['readout_length'] = READOUT_LENGTH
print(f'  Saved: {fine_out}')
print()
print('--- Leakage at channel boundaries ---')
print(f'  {"Window":<14}  {"at +0.5 ch":>12}  {"at +1.0 ch":>12}  {"at +2.0 ch":>12}')
print('  ' + '-'*54)
for wname, wdict in FINE_WIN_DEFS.items():
    r = fine_response[wname]
    def val_at(offset_ch):
        idx = np.argmin(np.abs(FINE_OFFSETS - offset_ch))
        return float(r[idx])
    print(f'  {wname:<14}  {val_at(0.5):>12.1f}  {val_at(1.0):>12.1f}  '
          f'{val_at(2.0):>12.1f}')
print()
print('  Theory (Rect/Hann/Black at +0.5ch): -3.7 / -1.4 / -1.0 dB')
print('  Theory (Rect/Hann/Black at +1.0ch): -13.3 / -31.5 / -58.1 dB')
print()
print('  READOUT_LENGTH restored to', READOUT_LENGTH)
print('  Saved fine-step npy arrays for all three windows')
print()
print('PASS - fine-step window response complete')


## Cell 10e — Multi-channel sinc response measurement (Jordan's plan)
Sweeps the DAC in 2000 fine steps across ONE channel width.
At each step: measures power in channels n=−2, −1, 0, +1, +2 simultaneously.
Rectangular window only.

**What this shows:** the sinc² response function of the rectangular window —
each channel traces a sinc² curve as the tone sweeps across it.
The crossing points between adjacent curves mark the channel boundaries.

**Fit:** fits a sinc² model to the n=0 curve to extract:
- The true bin centre frequency (as a freq2reg register value)
- The offset between the theoretical and measured bin centre
- The channel width as seen by the hardware

**Run AFTER Cells 5 and 6. Self-contained.**

In [ ]:
# ── Multi-channel sinc response (Jordan's plan) ───────────────────────────────
# X axis: CW tone frequency (DAC stepping)
# Y axis: power (dB) in channels n = -2, -1, 0, +1, +2
# Window: Rectangular only
# Steps: 2000 across one channel width (~0.28 kHz/step)
# Fit: sinc² to n=0 curve → true bin centre offset

SINC_CENTRE_MHZ  = 150.5    # must sit clearly inside one bin — 150.5 MHz lands in bin 270 (centre ~150.63 MHz)
SINC_N_STEPS     = 2000     # steps across 1 channel (Jordan: 'couple of thousands')
SINC_FRAMES      = 50       # frames per step (balance speed vs noise)
SINC_RL          = 993   # 993+2=995 (odd) — DMA delivers 995 cleanly
SINC_N_CHANNELS  = 2        # monitor ±N_CHANNELS from target (so n=-2,-1,0,+1,+2)

# Recompute DF from SINC_RL — critical: prevents stale kernel DF

# DAC and channel parameters
DAC_FS_MHZ  = soc.config['gens'][DAC_GEN]['fs']   # 9830.4 MHz
# Recompute channel width from SINC_RL — do not rely on outer-scope DF_MHZ
# This prevents stale kernel state from carrying over from previous cells
REG_BITS    = 32
SINC_DF     = FS_DECIMATED_MHZ / SINC_RL           # channel width
sinc_fax    = np.fft.rfftfreq(SINC_RL, d=1.0/(FS_DECIMATED_MHZ*1e6)) / 1e6

# Find target bin: use floor(f/df) so BIN_CENTRE = (K+0.5)*df
# is the TRUE centre of the bin CONTAINING SINC_CENTRE_MHZ.
# Bug fix: argmin on sinc_fax finds the left edge nearest target,
# causing BIN_CENTRE to be offset by +0.5 channels.
TARGET_K    = int(np.floor(SINC_CENTRE_MHZ / SINC_DF))
BIN_CENTRE  = (TARGET_K + 0.5) * SINC_DF
SWEEP_START = BIN_CENTRE - SINC_DF / 2
SWEEP_STOP  = BIN_CENTRE + SINC_DF / 2
SWEEP_FREQS = np.linspace(SWEEP_START, SWEEP_STOP, SINC_N_STEPS)

# Convert to actual freq2reg output frequencies
actual_freqs = np.array([round(f / DAC_FS_MHZ * 2**REG_BITS) *
                          DAC_FS_MHZ / 2**REG_BITS for f in SWEEP_FREQS])
offsets_ch   = (actual_freqs - BIN_CENTRE) / SINC_DF   # in channel units

# Channel bins to monitor: n = -N_CHANNELS ... +N_CHANNELS
monitor_bins = list(range(TARGET_K - SINC_N_CHANNELS,
                           TARGET_K + SINC_N_CHANNELS + 1))
monitor_ns   = list(range(-SINC_N_CHANNELS, SINC_N_CHANNELS + 1))

print('--- Multi-channel sinc response ---')
print(f'  Target bin     : {TARGET_K}  (theoretical centre = {BIN_CENTRE:.4f} MHz)')
print(f'  Channel width  : {SINC_DF*1e3:.4f} kHz')
print(f'  Sweep range    : {SWEEP_START:.4f} to {SWEEP_STOP:.4f} MHz')
print(f'  Steps          : {SINC_N_STEPS}  ({(SWEEP_STOP-SWEEP_START)*1e3/(SINC_N_STEPS-1):.3f} kHz/step)')
print(f'  DAC resolution : {DAC_FS_MHZ/2**REG_BITS*1e3:.4f} kHz/register step')
print(f'  Unique DAC freqs: {len(np.unique(actual_freqs))}  (many steps share same register)')
print(f'  Monitoring bins: {monitor_bins}  (n = {monitor_ns})')
print(f'  Window         : Rectangular')
print()

# Rectangular window
w_rect  = np.ones(SINC_RL, dtype=np.float32)
wp_rect = float(np.sum(w_rect**2))

# Build NoToneProgram
_sinc_no_cfg  = {'ro_ch': ADC_CH, 'readout_length': SINC_RL,
                 'adc_trig_offset': 200, 'soft_avgs': 1,
                 'reps': 1, 'relax_delay': 1.0}
_sinc_no_prog = NoToneProgram(soc, _sinc_no_cfg)

# Step 1: baseline
print('[STEP 1] Capturing 500 baseline frames...')
bl_acc = np.zeros(len(monitor_bins), dtype=np.float64)
for _ in range(500):
    iq    = _sinc_no_prog.acquire_decimated(soc, load_pulses=False, progress=False)
    i_raw = np.array(iq[0][0], dtype=np.float32)
    if len(i_raw) < SINC_RL:
        i_raw = np.pad(i_raw, (0, SINC_RL - len(i_raw)))
    pwr = np.abs(np.fft.rfft(i_raw[:SINC_RL] * w_rect))**2 / wp_rect
    for j, b in enumerate(monitor_bins):
        bl_acc[j] += pwr[b]
bl_power = bl_acc / 500
print(f'  Baseline captured for {len(monitor_bins)} channels')

# Step 2: fine sweep
print('[STEP 2] Running fine sweep...')
chan_power = np.zeros((SINC_N_STEPS, len(monitor_bins)), dtype=np.float64)
_sinc_cfg_base = dict(_TONE_CFG_BASE)
_sinc_cfg_base['readout_length'] = SINC_RL   # must match baseline RL

prev_reg = None
prog     = None
for step_idx, f_mhz in enumerate(actual_freqs):
    # Only create new program if DAC register changed
    reg = round(f_mhz / DAC_FS_MHZ * 2**REG_BITS)
    if reg != prev_reg:
        if prog is not None:
            disable_tone()
        cfg             = dict(_sinc_cfg_base)
        cfg['freq_mhz'] = f_mhz
        prog            = CWToneProgram(soc, cfg)
        prev_reg        = reg
        load_first      = True

    acc = np.zeros(len(monitor_bins), dtype=np.float64)
    for k in range(SINC_FRAMES):
        iq    = prog.acquire_decimated(soc, load_pulses=(k==0 and load_first),
                                       progress=False)
        load_first = False
        i_raw = np.array(iq[0][0], dtype=np.float32)
        if len(i_raw) < SINC_RL:
            i_raw = np.pad(i_raw, (0, SINC_RL - len(i_raw)))
        pwr = np.abs(np.fft.rfft(i_raw[:SINC_RL] * w_rect))**2 / wp_rect
        for j, b in enumerate(monitor_bins):
            acc[j] += pwr[b]

    for j in range(len(monitor_bins)):
        sig = max(acc[j]/SINC_FRAMES - bl_power[j], 1e-30)
        chan_power[step_idx, j] = 10.0 * math.log10(sig)

    if (step_idx+1) % 200 == 0 or step_idx == 0:
        print(f'  Step {step_idx+1:4d}/{SINC_N_STEPS}  '
              f'{f_mhz:.5f} MHz  offset={offsets_ch[step_idx]:+.4f} ch  '
              f'n=0: {chan_power[step_idx, SINC_N_CHANNELS]:.1f} dB  '
              f'n=+1: {chan_power[step_idx, SINC_N_CHANNELS+1]:.1f} dB')

disable_tone()

# Normalise to peak of n=0 channel
peak_n0 = float(np.max(chan_power[:, SINC_N_CHANNELS]))
chan_power_norm = chan_power - peak_n0

# ── Fit sinc² to n=0 channel ──────────────────────────────────────────────────
from scipy.optimize import curve_fit

def sinc2_model(x, d0, A, noise_floor):
    """sinc²(x - d0) model. x in channel units, d0 = true bin centre offset."""
    arg = np.pi * (x - d0)
    s   = np.where(np.abs(arg) < 1e-10, 1.0, np.sin(arg) / arg)
    return A * s**2 + noise_floor

n0_power  = chan_power[:, SINC_N_CHANNELS]
# Fit only where signal is well above noise (top 30 dB of range)
fit_mask  = n0_power > (peak_n0 - 30)
try:
    popt, pcov = curve_fit(sinc2_model, offsets_ch[fit_mask],
                           10**(n0_power[fit_mask]/10),
                           p0=[0.0, 10**(peak_n0/10), 0.0],
                           maxfev=10000)
    d0_fit    = popt[0]
    perr      = np.sqrt(np.diag(pcov))
    d0_err    = perr[0]
    fit_ok    = True
    # True bin centre frequency
    true_centre_mhz = BIN_CENTRE + d0_fit * SINC_DF
    true_centre_reg = round(true_centre_mhz / DAC_FS_MHZ * 2**REG_BITS)
    true_centre_actual = true_centre_reg * DAC_FS_MHZ / 2**REG_BITS
except Exception as e:
    fit_ok = False
    print(f'  Fit failed: {e}')
    d0_fit = 0.0

# ── Plot ──────────────────────────────────────────────────────────────────────
colors = ['#9b59b6', '#e74c3c', '#2ecc71', '#3498db', '#f39c12']
labels = [f'n={n}' for n in monitor_ns]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Top: all channel responses
for j, (n, col, lab) in enumerate(zip(monitor_ns, colors, labels)):
    lw = 2.5 if n == 0 else 1.5
    ax1.plot(offsets_ch, chan_power_norm[:, j],
             color=col, lw=lw, label=lab, alpha=0.9)

# Theoretical sinc² overlays
d_th = np.linspace(-0.5, 0.5, 2000)
for n, col in zip(monitor_ns, colors):
    arg = np.pi * (d_th - n)
    s   = np.where(np.abs(arg) < 1e-10, 1.0, np.sin(arg)/arg)
    th  = 20 * np.log10(np.abs(s) + 1e-15)
    ax1.plot(d_th, th, color=col, lw=0.6, linestyle='--', alpha=0.35)

if fit_ok:
    ax1.axvline(d0_fit, color='yellow', lw=2.0, linestyle='-.',
                label=f'Fitted centre offset: {d0_fit:+.4f} ch\n'
                      f'True centre: {true_centre_actual:.5f} MHz')

ax1.axvline(0, color='white', lw=0.8, linestyle='--', alpha=0.4,
            label='Theoretical bin centre')
ax1.axvline(-0.5, color='grey', lw=0.8, linestyle=':', alpha=0.5)
ax1.axvline( 0.5, color='grey', lw=0.8, linestyle=':', alpha=0.5,
             label='Channel boundaries (±0.5 ch)')
ax1.set_ylabel('Power (dB, normalised to n=0 peak)', fontsize=11)
ax1.set_title('Rectangular window sinc² response — power in each channel\n'
              'Bold lines = measured,  faint dashed = theoretical sinc²',
              fontsize=11)
ax1.legend(fontsize=8, ncol=3)
ax1.grid(True, alpha=0.2)
ax1.set_ylim(-80, 5)

# Bottom: n=0 with sinc² fit
ax2.plot(offsets_ch, chan_power_norm[:, SINC_N_CHANNELS],
         color='#2ecc71', lw=2.0, label='n=0 measured')
if fit_ok:
    d_fit_x = np.linspace(-0.5, 0.5, 2000)
    fit_y   = sinc2_model(d_fit_x, *popt)
    fit_dB  = 10 * np.log10(np.maximum(fit_y / popt[1], 1e-30))
    ax2.plot(d_fit_x, fit_dB, color='yellow', lw=2.0, linestyle='--',
             label=f'sinc² fit  d0={d0_fit:+.4f} ch ({d0_fit*SINC_DF*1e3:+.2f} kHz)')
    ax2.axvline(d0_fit, color='yellow', lw=1.5, linestyle=':')
ax2.axvline(0, color='white', lw=0.8, linestyle='--', alpha=0.4)
ax2.set_xlabel('CW tone offset from theoretical bin centre (channels)', fontsize=11)
ax2.set_ylabel('n=0 channel power (dB, normalised)', fontsize=11)
ax2.set_title('sinc² fit to n=0 channel — extracts true bin centre offset',
              fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.2)
ax2.set_ylim(-60, 5)

ts_sinc = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
fig.suptitle(
    f'RHINO RFSoC 4x2 — Multi-channel sinc response  |  Rectangular window\n'
    f'Target bin {TARGET_K} ({BIN_CENTRE:.4f} MHz)  |  '
    f'{SINC_N_STEPS} steps  |  {SINC_FRAMES} frames/step',
    fontsize=12
)
plt.tight_layout()
sinc_out = f'{SAVE_DIR}sinc_response_{ts_sinc}.png'
plt.savefig(sinc_out, dpi=150, bbox_inches='tight')
plt.show()

# Save
np.save(f'{SAVE_DIR}sinc_freqs_{ts_sinc}.npy',       actual_freqs)
np.save(f'{SAVE_DIR}sinc_offsets_{ts_sinc}.npy',      offsets_ch)
np.save(f'{SAVE_DIR}sinc_chan_power_{ts_sinc}.npy',   chan_power)

_TONE_CFG_BASE['readout_length'] = READOUT_LENGTH
print(f'\n  Saved: {sinc_out}')
print()
if fit_ok:
    print(f'=== FITTED BIN CENTRE ===')
    print(f'  Offset from theoretical: {d0_fit:+.5f} channels  '
          f'({d0_fit*SINC_DF*1e3:+.3f} kHz)')
    print(f'  True bin centre        : {true_centre_mhz:.5f} MHz')
    print(f'  Best DAC register      : {true_centre_reg}')
    print(f'  Best DAC frequency     : {true_centre_actual:.5f} MHz')
    print(f'  Residual offset        : '
          f'{(true_centre_actual - true_centre_mhz)*1e3:.4f} kHz')
    print()
    print('  Use this register value in future sweeps for minimum')
    print('  leakage into neighbouring channels (Rectangular window).')
print()
_TONE_CFG_BASE['readout_length'] = READOUT_LENGTH
print('  READOUT_LENGTH restored to', READOUT_LENGTH)
print()
print('PASS - multi-channel sinc response complete')


## Cell 11b — Amplitude sweep (absolute gain measurement)
Varies `DAC_AMPLITUDE` at a single frequency to extract the absolute gain g(ν).
The ADC power should increase linearly with DAC power: `P_ADC = g(ν) · P_DAC + P_noise`.
The slope of the linear fit is g(ν) at that frequency.

**Run AFTER Cell 6 (CWToneProgram defined) and Cell 7 (baseline measured).**
Change `AMP_SWEEP_FREQ_MHZ` to the frequency you want to characterise.
Avoid the DAC null region (93–111 MHz).

In [ ]:
# ── Amplitude sweep — absolute gain at one frequency ─────────────────────────
# Sweeps DAC_AMPLITUDE from 0.1 to 1.0, measures peak ADC power at each level.
# Linear fit: P_ADC_linear = g(nu) * P_DAC_linear + P_noise
# g(nu) = slope of P_ADC vs P_DAC → absolute gain of the signal chain.

AMP_SWEEP_FREQ_MHZ = 150.0   # ← choose a frequency away from DAC nulls
AMP_SWEEP_LEVELS   = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
AMP_SWEEP_FRAMES   = 50      # frames to average at each amplitude level

print(f'[AMP SWEEP] Frequency: {AMP_SWEEP_FREQ_MHZ} MHz')
print(f'  Amplitudes : {AMP_SWEEP_LEVELS}')
print(f'  Frames/amp : {AMP_SWEEP_FRAMES}')
print(f'  DAC null region (93-111 MHz) — avoid setting AMP_SWEEP_FREQ_MHZ there')

eb_amp = int(np.argmin(np.abs(freq_axis_mhz - AMP_SWEEP_FREQ_MHZ)))

amp_peak_power   = []   # peak ADC power (dB) at each amplitude
amp_dac_power_dB = []   # estimated DAC output power (dBm) at each amplitude

for amp in AMP_SWEEP_LEVELS:
    set_cw_tone(AMP_SWEEP_FREQ_MHZ, amplitude=amp)
    acc = np.zeros(len(freq_axis_mhz), dtype=np.float64)
    for _ in range(AMP_SWEEP_FRAMES):
        acc += capture_tone_sample().astype(np.float64)
    avg = (acc / AMP_SWEEP_FRAMES).astype(np.float32)
    disable_tone()

    lo = max(0, eb_amp - 3)
    hi = min(len(avg), eb_amp + 4)
    peak_db = float(np.max(avg[lo:hi]))   # dB above baseline
    amp_peak_power.append(peak_db)

    # Estimated DAC power (dBm) — rough, assumes 1Vpp FS into 50 Ohm
    V_rms = (amp * 1.0) / (2 * math.sqrt(2))
    P_dBm = 10 * math.log10(max(V_rms**2 / 50.0 / 1e-3, 1e-20))
    amp_dac_power_dB.append(P_dBm)
    print(f'  amp={amp:.1f}  DAC≈{P_dBm:.1f} dBm  ADC_peak={peak_db:.1f} dB')

amp_peak_power   = np.array(amp_peak_power)
amp_dac_power_dB = np.array(amp_dac_power_dB)

# ── Linear fit in linear (not dB) units ──────────────────────────────────────
# Convert dB above noise floor to linear power ratio
amp_peak_linear = 10.0**(amp_peak_power / 10.0)
amp_dac_linear  = 10.0**(amp_dac_power_dB / 10.0)

# Fit P_ADC = g * P_DAC + c  (linear in linear units)
A = np.column_stack([amp_dac_linear, np.ones_like(amp_dac_linear)])
result = np.linalg.lstsq(A, amp_peak_linear, rcond=None)
g_linear, c_linear = result[0]
g_dB = 10 * math.log10(max(g_linear, 1e-30))

# R² of the linear fit
predicted = g_linear * amp_dac_linear + c_linear
ss_res = float(np.sum((amp_peak_linear - predicted)**2))
ss_tot = float(np.sum((amp_peak_linear - np.mean(amp_peak_linear))**2))
r2     = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0

print(f'\n--- Amplitude sweep results at {AMP_SWEEP_FREQ_MHZ} MHz ---')
print(f'  g(ν)  = {g_linear:.4f} (linear)  =  {g_dB:.2f} dB')
print(f'  offset = {c_linear:.4f} (noise floor offset)')
print(f'  R²    = {r2:.4f}  (1.0 = perfect linear response)')
if r2 > 0.99:
    print('  ✅ Excellent linearity — DAC and ADC both in linear regime')
elif r2 > 0.95:
    print('  ⚠️  Moderate linearity — check if high amplitudes saturate the ADC')
else:
    print('  ❌ Poor linearity — ADC may be saturating at high amplitudes')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: dB vs dB
ax1.plot(amp_dac_power_dB, amp_peak_power, 'o-', color='steelblue',
         markersize=6, label='Measured')
ax1.set_xlabel('Estimated DAC output power (dBm)', fontsize=10)
ax1.set_ylabel('ADC peak power above noise floor (dB)', fontsize=10)
ax1.set_title(f'Amplitude sweep — {AMP_SWEEP_FREQ_MHZ} MHz  |  dB vs dB view', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=9)

# Right: linear units with fit line
x_fit = np.linspace(amp_dac_linear.min(), amp_dac_linear.max(), 100)
y_fit = g_linear * x_fit + c_linear
ax2.plot(amp_dac_linear, amp_peak_linear, 'o', color='steelblue',
         markersize=6, label='Measured')
ax2.plot(x_fit, y_fit, '-', color='orange', lw=1.5,
         label=f'Linear fit  g={g_dB:.2f} dB  R²={r2:.4f}')
ax2.set_xlabel('Estimated DAC power (linear, mW)', fontsize=10)
ax2.set_ylabel('ADC peak power (linear ratio)', fontsize=10)
ax2.set_title(f'Amplitude sweep — {AMP_SWEEP_FREQ_MHZ} MHz  |  Linear fit for g(v)', fontsize=10)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

fig.suptitle(
    f'RHINO RFSoC 4x2 — Amplitude sweep at {AMP_SWEEP_FREQ_MHZ} MHz  |  '
    f'g(v) = {g_dB:.2f} dB  |  R2 = {r2:.4f}  |  {AMP_SWEEP_FRAMES} frames/level',
    fontsize=11
)
plt.tight_layout()
amp_out = f'{SAVE_DIR}amp_sweep_{AMP_SWEEP_FREQ_MHZ:.0f}MHz_{datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")}.png'
plt.savefig(amp_out, dpi=150, bbox_inches='tight')
plt.show()

# Save results
np.save(amp_out.replace('.png', '_amplitudes.npy'), np.array(AMP_SWEEP_LEVELS))
np.save(amp_out.replace('.png', '_adc_power_dB.npy'), amp_peak_power)
np.save(amp_out.replace('.png', '_dac_power_dBm.npy'), amp_dac_power_dB)
print(f'\n  Saved: {amp_out}')
print('\n✅ PASS — amplitude sweep complete')


## Cell 11 — Tone SNR summary table
Prints a summary table of the measured tone SNR at each DAC step.  
This is the key diagnostic — each step should show clear positive SNR  
at the expected frequency.

In [ ]:
# Compute avg_img if Cell 10 (averaged waterfall) was skipped
if 'avg_img' not in dir():
    n_freq  = len(freq_axis_mhz)
    avg_img = np.zeros((N_STEPS, n_freq), dtype=np.float32)
    for _i, _r in enumerate(dac_step_rows):
        avg_img[_i] = np.mean(Spectra[_r:_r+N_FRAMES_PER_STEP], axis=0)
    print('[INFO] avg_img computed from Spectra (Cell 10 was skipped)')

print('=' * 70)
print('CW CALIBRATION RESULTS — TONE SNR PER STEP')
print(f'  Sweep: {SWEEP_BAND_START:.3f}–{SWEEP_BAND_END:.3f} MHz  |  '
      f'{N_FRAMES_PER_STEP} frames/step  |  {N_STEPS} steps')
print(f'  Tone formula (QICK): f_tone = f_DAC  (no F_LO offset)')
print('=' * 70)
print(f'  {"Step":>5}  {"DAC (MHz)":>10}  {"Exp. bin":>9}  '
      f'{"Peak bin":>9}  {"Peak MHz":>10}  {"SNR (dB)":>10}  {"Pass?"}')
print('  ' + '-' * 68)

n_pass = 0
for i, f_mhz in enumerate(DAC_FREQS_MHZ):
    expected_bin = int(np.argmin(np.abs(freq_axis_mhz - f_mhz)))

    lo = max(0, expected_bin - 5)
    hi = min(n_freq, expected_bin + 6)
    peak_offset = int(np.argmax(avg_img[i, lo:hi]))
    actual_bin  = lo + peak_offset
    actual_mhz  = float(freq_axis_mhz[actual_bin])
    snr         = float(avg_img[i, actual_bin])
    flagged     = bool(spur_mask[i]) if 'spur_mask' in dir() else False
    passed      = (snr > 5.0 and abs(actual_mhz - f_mhz) < 2.0) and not flagged

    if passed:
        n_pass += 1

    sym = '⚠️ ' if flagged else ('✅' if passed else '❌')
    print(f'  {i+1:>5d}  {f_mhz:>10.2f}  {expected_bin:>9d}  '
          f'{actual_bin:>9d}  {actual_mhz:>10.2f}  {snr:>10.1f}  {sym}')

print('  ' + '-' * 68)
print(f'\n  Pass rate: {n_pass}/{N_STEPS} steps with SNR > 5 dB and frequency error < 2 MHz')
print()
if n_pass == N_STEPS:
    print('   ALL STEPS PASS — gain curve complete')
elif n_pass > N_STEPS * 0.8:
    print(f'    {N_STEPS - n_pass} steps failed — check for RFI at those frequencies')
else:
    print(f'   {N_STEPS - n_pass} steps failed — check loopback cable and DAC amplitude')

## Cell 12 — Save all results
Saves the complete dataset and prints download instructions.  
This is everything needed for the RASTI paper results section.

In [ ]:
# ts is defined in Cell 8 (waterfall acquisition). Guard in case it wasn't run.
if 'ts' not in dir():
    ts = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
    print('[INFO] ts not found from Cell 8 — using current time for filenames')

print('[SAVE] Saving final results...')

files_saved = {
    f'cw_waterfall_{ts}.npy'     : 'Full waterfall array (N_TOTAL_FRAMES × n_bins)',
    f'cw_avg_waterfall_{ts}.npy' : 'Averaged waterfall (N_STEPS × n_bins)',
    f'cw_gain_curve_{ts}.npy'    : 'Gain curve g(ν) — one value per DAC step',
    f'cw_dac_freqs_{ts}.npy'     : 'DAC frequencies (MHz) — N_STEPS values',
    f'cw_freq_axis_{ts}.npy'     : 'Frequency axis (MHz) — n_bins values',
    f'cw_baseline.npy'           : 'Noise floor baseline (n_bins values)',
    f'cw_waterfall_plot_{ts}.png': 'Full waterfall plot',
    f'cw_gain_curve_{ts}.png'    : 'Averaged waterfall + gain curve plot',
}

for fname, desc in files_saved.items():
    fpath = f'{SAVE_DIR}{fname}'
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f'   {fname:<40} ({size_mb:.1f} MB)  — {desc}')
    else:
        print(f'    {fname:<40} NOT FOUND')

print(f'\n--- Reload these results on any machine ---')
print(f'import numpy as np')
print(f'Spectra    = np.load("{SAVE_DIR}cw_waterfall_{ts}.npy")')
print(f'avg_img    = np.load("{SAVE_DIR}cw_avg_waterfall_{ts}.npy")')
print(f'gain_curve = np.load("{SAVE_DIR}cw_gain_curve_{ts}.npy")')
print(f'dac_freqs  = np.load("{SAVE_DIR}cw_dac_freqs_{ts}.npy")')
print(f'freq_axis  = np.load("{SAVE_DIR}cw_freq_axis_{ts}.npy")')
print()
print(f'--- Download from board ---')
print(f'scp xilinx@192.168.2.99:{SAVE_DIR}*_{ts}* .')
print()
print(' DONE — CW calibration experiment complete')
print()
print('Next steps:')
print('  1. Vary DAC amplitude (0.1–1.0) at each frequency step')
print('     to extract g(ν) via the linear fit P_ADC = g(ν)·P_CW + P_noise')
print('  2. Replace loopback with signal generator for absolute power reference')
print('  3. Compare gain curve with rfsoc_sam result (where available)')